# Module 07: Optimizers - Sophisticated Learning Algorithms

Welcome to Module 07! You'll build optimizers that enable neural networks to learn from gradients using sophisticated algorithms.

In [1]:
#| default_exp optimizers
#| export

import numpy as np
rng = np.random.default_rng(7)
from typing import List, Union, Optional, Dict, Any

import sys
sys.path.append(r"D:\DS\Data Science\TinyTorch\projects\my-tinytorch-systems")

# Import Tensor from Module 01 (now with gradient support from Module 06)
from tinytorch.foundation.tensor import Tensor

# Enable autograd to add gradient tracking to Tensor
# This module depends on Module 06 (Autograd) being available
from tinytorch.foundation.autograd import enable_autograd
enable_autograd()

# Constants for optimizer defaults
DEFAULT_LEARNING_RATE_SGD = 0.01  # Default learning rate for SGD
DEFAULT_LEARNING_RATE_ADAM = 0.001  # Default learning rate for Adam/AdamW
DEFAULT_MOMENTUM = 0.9  # Default momentum for SGD
DEFAULT_BETA1 = 0.9  # First moment decay rate for Adam
DEFAULT_BETA2 = 0.999  # Second moment decay rate for Adam
DEFAULT_EPS = 1e-8  # Small epsilon for numerical stability in Adam
DEFAULT_WEIGHT_DECAY_ADAMW = 0.01  # Default weight decay for AdamW

In [2]:
#| export
class Optimizer:
    """
    Base class for all optimizers.
    """

    def __init__(self, params: List[Tensor]):
        """
        Initialize optimizer with parameters to optimize.
        """
        
        # Validate and store parameters
        if not isinstance(params, list):
            params = list(params)

        # Store parameters
        self.params = params

        # Ensure parameters participate in autograd once it is enabled
        for param in self.params:
            if isinstance(param, Tensor):
                param.requires_grad = True
                param.grad = None
        self.step_count = 0  # For algorithms that need step counting
        

    def zero_grad(self):
        """
        Clear gradients from all parameters.

        """
        
        for param in self.params:
            param.grad = None
        

    def step(self):
        """
        Update parameters based on gradients.

        This is abstract - each optimizer implements its own update rule.
        """
        raise NotImplementedError(
            f"Abstract method step() not implemented\n"
            f"  ❌ {self.__class__.__name__} inherits from Optimizer but doesn't define step()\n"
            f"  💡 Each optimizer must implement its own update rule (SGD, Adam, etc.)\n"
            f"  🔧 Override step() in your optimizer subclass:\n"
            f"      def step(self):\n"
            f"          for param in self.params:\n"
            f"              if param.grad is not None:\n"
            f"                  param.data -= self.lr * param.grad.data"
        )

In [3]:
#| export
class _ExtractGradientMixin:
    """Mixin added to Optimizer for gradient extraction."""
    def _extract_gradient(self, param: Tensor) -> np.ndarray:
        """
        Extract gradient data as a NumPy array from a parameter.
        """
        
        grad = param.grad
        if isinstance(grad, Tensor):
            return grad.data
        else:
            return grad
        

# Attach _extract_gradient to Optimizer so all subclasses inherit it
Optimizer._extract_gradient = _ExtractGradientMixin._extract_gradient

In [4]:
def test_unit_extract_gradient():
    """ Test _extract_gradient handles Tensor and ndarray gradients."""
    print(" Unit Test: Gradient Extraction...")

    param = Tensor([1.0, 2.0], requires_grad=True)
    optimizer = Optimizer([param])

    # Case 1: Gradient is a Tensor
    param.grad = Tensor([0.1, 0.2])
    grad_data = optimizer._extract_gradient(param)
    assert isinstance(grad_data, np.ndarray), "Should return ndarray from Tensor grad"
    assert np.allclose(grad_data, [0.1, 0.2])

    # Case 2: Gradient is a raw NumPy array
    param.grad = np.array([0.3, 0.4])
    grad_data = optimizer._extract_gradient(param)
    assert isinstance(grad_data, np.ndarray), "Should return ndarray from ndarray grad"
    assert np.allclose(grad_data, [0.3, 0.4])

    # Case 3: Multi-dimensional gradient
    param_2d = Tensor([[1.0, 2.0], [3.0, 4.0]], requires_grad=True)
    opt_2d = Optimizer([param_2d])
    param_2d.grad = Tensor([[0.1, 0.2], [0.3, 0.4]])
    grad_data_2d = opt_2d._extract_gradient(param_2d)
    assert grad_data_2d.shape == (2, 2)
    assert np.allclose(grad_data_2d, [[0.1, 0.2], [0.3, 0.4]])

    print("✅ Gradient extraction works correctly!")

if __name__ == "__main__":
    test_unit_extract_gradient()

 Unit Test: Gradient Extraction...
✅ Gradient extraction works correctly!


In [5]:
def test_unit_optimizer_base():
    """ Test base Optimizer functionality."""
    print(" Unit Test: Base Optimizer...")

    # Create test parameters
    param1 = Tensor([1.0, 2.0], requires_grad=True)
    param2 = Tensor([[3.0, 4.0], [5.0, 6.0]], requires_grad=True)

    # Create optimizer first (optimizer.__init__ resets grad to None)
    optimizer = Optimizer([param1, param2])

    # Test parameter storage
    assert len(optimizer.params) == 2
    assert optimizer.params[0] is param1
    assert optimizer.params[1] is param2
    assert optimizer.step_count == 0

    # Add gradients AFTER creating optimizer to test zero_grad properly
    param1.grad = Tensor([0.1, 0.2])
    param2.grad = Tensor([[0.3, 0.4], [0.5, 0.6]])

    # Test zero_grad
    optimizer.zero_grad()
    assert param1.grad is None
    assert param2.grad is None

    # Test that optimizer accepts any tensor (no validation required)
    # Gradient tracking is handled by the autograd module
    regular_param = Tensor([1.0])
    opt = Optimizer([regular_param])
    assert len(opt.params) == 1

    print("✅ Base Optimizer works correctly!")

if __name__ == "__main__":
    test_unit_optimizer_base()

 Unit Test: Base Optimizer...
✅ Base Optimizer works correctly!


##  SGD - Stochastic Gradient Descent

In [6]:
#| export
class SGD(Optimizer):
    """
    Stochastic Gradient Descent with momentum.
    """

    def __init__(self, params: List[Tensor], lr: float = DEFAULT_LEARNING_RATE_SGD, momentum: float = 0.0, weight_decay: float = 0.0):
        """
        Initialize SGD optimizer.

        """
        
        super().__init__(params)

        self.lr = lr
        self.momentum = momentum
        self.weight_decay = weight_decay

        # Initialize momentum buffers (created lazily)
        self.momentum_buffers = [None for _ in self.params]
        

    def has_momentum(self) -> bool:
        """
        Check if this optimizer uses momentum.
        """
        return self.momentum > 0

    def get_momentum_state(self) -> Optional[List]:
        """
        Get momentum buffers for checkpointing.
        """
        if not self.has_momentum():
            return None
        return [buf.copy() if buf is not None else None
                for buf in self.momentum_buffers]

    def set_momentum_state(self, state: Optional[List]) -> None:
        """
        Restore momentum buffers from checkpointing.
        """
        if state is None or not self.has_momentum():
            return

        if len(state) != len(self.momentum_buffers):
            raise ValueError(
                f"Momentum state length mismatch\n"
                f"  State has {len(state)} buffers, but optimizer has {len(self.momentum_buffers)} parameters\n"
                f"  Checkpoint was saved with a different model architecture or parameter count\n"
                f"  Ensure you're loading state into an optimizer with the same number of parameters:\n"
                f"   # Check parameter counts match before restoring\n"
                f"   assert len(saved_state) == len(optimizer.params)"
            )

        for i, buf in enumerate(state):
            if buf is not None:
                self.momentum_buffers[i] = buf.copy()

    def step(self):
        """
        Perform SGD update step with momentum.

        """
        
        for i, param in enumerate(self.params):
            if param.grad is None:
                continue

            # Extract gradient using shared helper
            grad_data = self._extract_gradient(param)

            # Apply weight decay
            if self.weight_decay != 0:
                grad_data = grad_data + self.weight_decay * param.data

            # Update momentum buffer
            if self.momentum != 0:
                if self.momentum_buffers[i] is None:
                    # Initialize momentum buffer
                    self.momentum_buffers[i] = np.zeros_like(param.data)

                # Update momentum: v = momentum * v_prev + grad
                self.momentum_buffers[i] = self.momentum * self.momentum_buffers[i] + grad_data
                grad_data = self.momentum_buffers[i]

            # Update parameter: param = param - lr * grad
            param.data = param.data - self.lr * grad_data

        # Increment step counter
        self.step_count += 1
        

###  Unit Test: SGD Optimizer

This test validates our SGD implementation works correctly.

In [7]:
def test_unit_sgd_optimizer():
    """ Test SGD optimizer implementation."""
    print(" Unit Test: SGD Optimizer...")

    # Test basic SGD without momentum
    param = Tensor([1.0, 2.0], requires_grad=True)
    optimizer = SGD([param], lr=0.1)
    # Set gradient AFTER creating optimizer (optimizer.__init__ resets grad to None)
    param.grad = Tensor([0.1, 0.2])
    original_data = param.data.copy()

    optimizer.step()

    # Expected: param = param - lr * grad = [1.0, 2.0] - 0.1 * [0.1, 0.2] = [0.99, 1.98]
    expected = original_data - 0.1 * np.array([0.1, 0.2])
    assert np.allclose(param.data, expected)
    assert optimizer.step_count == 1

    # Test SGD with momentum
    param2 = Tensor([1.0, 2.0], requires_grad=True)
    optimizer_momentum = SGD([param2], lr=0.1, momentum=0.9)
    # Set gradient AFTER creating optimizer
    param2.grad = Tensor([0.1, 0.2])

    # First step: v = 0.9 * 0 + [0.1, 0.2] = [0.1, 0.2]
    optimizer_momentum.step()
    expected_first = np.array([1.0, 2.0]) - 0.1 * np.array([0.1, 0.2])
    assert np.allclose(param2.data, expected_first)

    # Second step with same gradient
    param2.grad = Tensor([0.1, 0.2])
    optimizer_momentum.step()
    # v = 0.9 * [0.1, 0.2] + [0.1, 0.2] = [0.19, 0.38]
    expected_momentum = np.array([0.19, 0.38])
    expected_second = expected_first - 0.1 * expected_momentum
    assert np.allclose(param2.data, expected_second, rtol=1e-5)

    # Test weight decay
    param3 = Tensor([1.0, 2.0], requires_grad=True)
    optimizer_wd = SGD([param3], lr=0.1, weight_decay=0.01)
    # Set gradient AFTER creating optimizer
    param3.grad = Tensor([0.1, 0.2])
    optimizer_wd.step()

    # grad_with_decay = [0.1, 0.2] + 0.01 * [1.0, 2.0] = [0.11, 0.22]
    expected_wd = np.array([1.0, 2.0]) - 0.1 * np.array([0.11, 0.22])
    assert np.allclose(param3.data, expected_wd)

    print("✅ SGD optimizer works correctly!")

if __name__ == "__main__":
    test_unit_sgd_optimizer()

 Unit Test: SGD Optimizer...
✅ SGD optimizer works correctly!


## Adam - Adaptive Moment Estimation

In [9]:
#| export
class Adam(Optimizer):
    """
    Adam optimizer with adaptive learning rates.
    """

    def __init__(self, params: List[Tensor], lr: float = DEFAULT_LEARNING_RATE_ADAM, betas: tuple = (DEFAULT_BETA1, DEFAULT_BETA2), eps: float = DEFAULT_EPS, weight_decay: float = 0.0):
        """
        Initialize Adam optimizer.
        """
        
        super().__init__(params)

        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.weight_decay = weight_decay

        # Initialize moment buffers (created lazily)
        self.m_buffers = [None for _ in self.params]  # First moment (mean)
        self.v_buffers = [None for _ in self.params]  # Second moment (variance)
        

###  Moment Updates - EMA and Bias Correction

This helper isolates the EMA (exponential moving averages) + bias correction math so that `step()`
only has to compose: extract gradient, update moments, apply update.

In [10]:
#| export
class _AdamUpdateMomentsMixin:
    """Mixin added to Adam for moment updates."""
    def _update_moments(self, i: int, grad_data: np.ndarray) -> tuple:
        """
        Update first and second moment estimates with bias correction.
        """
        
        # Initialize buffers if needed
        if self.m_buffers[i] is None:
            self.m_buffers[i] = np.zeros_like(grad_data)
            self.v_buffers[i] = np.zeros_like(grad_data)

        # Update biased first moment estimate
        self.m_buffers[i] = self.beta1 * self.m_buffers[i] + (1 - self.beta1) * grad_data

        # Update biased second moment estimate
        self.v_buffers[i] = self.beta2 * self.v_buffers[i] + (1 - self.beta2) * (grad_data ** 2)

        # Compute bias correction
        bias_correction1 = 1 - self.beta1 ** self.step_count
        bias_correction2 = 1 - self.beta2 ** self.step_count

        # Compute bias-corrected moments
        m_hat = self.m_buffers[i] / bias_correction1
        v_hat = self.v_buffers[i] / bias_correction2

        return m_hat, v_hat
        

# Attach _update_moments to Adam
Adam._update_moments = _AdamUpdateMomentsMixin._update_moments

###  Unit Test: Adam Moment Updates

This test validates that `_update_moments` correctly computes exponential
moving averages and bias correction for Adam's first and second moments.

In [11]:
def test_unit_adam_update_moments():
    """ Test Adam _update_moments computes correct EMA and bias correction."""
    print(" Unit Test: Adam Moment Updates...")

    param = Tensor([1.0, 2.0], requires_grad=True)
    optimizer = Adam([param], lr=0.01, betas=(0.9, 0.999), eps=1e-8)

    grad = np.array([0.1, 0.2])

    # Simulate step 1 (step_count must be set before calling _update_moments)
    optimizer.step_count = 1

    m_hat, v_hat = optimizer._update_moments(0, grad)

    # Manual calculation for step 1:
    # m = 0.9 * 0 + 0.1 * [0.1, 0.2] = [0.01, 0.02]
    # v = 0.999 * 0 + 0.001 * [0.01, 0.04] = [0.00001, 0.00004]
    # bias_correction1 = 1 - 0.9^1 = 0.1
    # bias_correction2 = 1 - 0.999^1 = 0.001
    # m_hat = [0.01, 0.02] / 0.1 = [0.1, 0.2]  (= grad, as expected!)
    # v_hat = [0.00001, 0.00004] / 0.001 = [0.01, 0.04]  (= grad^2)

    assert np.allclose(m_hat, grad), f"m_hat should equal grad at step 1, got {m_hat}"
    assert np.allclose(v_hat, grad ** 2), f"v_hat should equal grad^2 at step 1, got {v_hat}"

    # Step 2 with same gradient
    optimizer.step_count = 2
    m_hat2, v_hat2 = optimizer._update_moments(0, grad)

    # Moments should still be close to grad (converging to true mean)
    assert m_hat2 is not None
    assert v_hat2 is not None
    # Buffers should be updated in-place
    assert optimizer.m_buffers[0] is not None
    assert optimizer.v_buffers[0] is not None

    print("✅ Adam moment updates work correctly!")

if __name__ == "__main__":
    test_unit_adam_update_moments()

 Unit Test: Adam Moment Updates...
✅ Adam moment updates work correctly!


###  Adam Step - Composing Gradient Extraction, Moments, and Update


In [12]:
#| export
class _AdamStepMixin:
    """Mixin added to Adam for step method."""
    def step(self):
        """
        Perform Adam update step by composing helpers.
        """
        
        # Increment step counter first (needed for bias correction)
        self.step_count += 1

        for i, param in enumerate(self.params):
            if param.grad is None:
                continue

            # Extract gradient using shared helper
            grad_data = self._extract_gradient(param)

            # Apply weight decay
            if self.weight_decay != 0:
                grad_data = grad_data + self.weight_decay * param.data

            # Update moments and get bias-corrected estimates
            m_hat, v_hat = self._update_moments(i, grad_data)

            # Update parameter
            param.data = param.data - self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

# Attach step to Adam
Adam.step = _AdamStepMixin.step

### Unit Test: Adam Optimizer

This test validates our Adam implementation works correctly.

In [14]:
def test_unit_adam_optimizer():
    """ Test Adam optimizer implementation."""
    print(" Unit Test: Adam Optimizer...")

    # Test basic Adam functionality
    param = Tensor([1.0, 2.0], requires_grad=True)
    optimizer = Adam([param], lr=0.01, betas=(0.9, 0.999), eps=1e-8)
    # Set gradient AFTER creating optimizer (optimizer.__init__ resets grad to None)
    param.grad = Tensor([0.1, 0.2])
    original_data = param.data.copy()

    # First step
    optimizer.step()

    # Manually compute expected values
    grad = np.array([0.1, 0.2])

    # First moment: m = 0.9 * 0 + 0.1 * grad = 0.1 * grad
    m = 0.1 * grad

    # Second moment: v = 0.999 * 0 + 0.001 * grad^2 = 0.001 * grad^2
    v = 0.001 * (grad ** 2)

    # Bias correction
    bias_correction1 = 1 - 0.9 ** 1  # = 0.1
    bias_correction2 = 1 - 0.999 ** 1  # = 0.001

    m_hat = m / bias_correction1  # = grad
    v_hat = v / bias_correction2  # = grad^2

    # Update
    expected = original_data - 0.01 * m_hat / (np.sqrt(v_hat) + 1e-8)

    assert np.allclose(param.data, expected, rtol=1e-6)
    assert optimizer.step_count == 1

    # Test second step to verify moment accumulation
    param.grad = Tensor([0.1, 0.2])
    optimizer.step()

    # Should have updated moments
    assert optimizer.m_buffers[0] is not None
    assert optimizer.v_buffers[0] is not None
    assert optimizer.step_count == 2

    # Test with weight decay
    param2 = Tensor([1.0, 2.0], requires_grad=True)
    optimizer_wd = Adam([param2], lr=0.01, weight_decay=0.01)
    # Set gradient AFTER creating optimizer
    param2.grad = Tensor([0.1, 0.2])
    optimizer_wd.step()

    # Weight decay should modify the effective gradient
    # grad_with_decay = [0.1, 0.2] + 0.01 * [1.0, 2.0] = [0.11, 0.22]
    # The exact computation is complex, but we can verify parameter changed
    assert not np.array_equal(param2.data, np.array([1.0, 2.0]))

    print("✅ Adam optimizer works correctly!")

if __name__ == "__main__":
    test_unit_adam_optimizer()

 Unit Test: Adam Optimizer...
✅ Adam optimizer works correctly!


##  AdamW - Adam with Decoupled Weight Decay

AdamW fixes a subtle but important bug in Adam's weight decay implementation. The bug affects how regularization interacts with adaptive learning rates.

In [15]:
#| export
class AdamW(Optimizer):
    """
    AdamW optimizer with decoupled weight decay.
    """

    def __init__(self, params: List[Tensor], lr: float = DEFAULT_LEARNING_RATE_ADAM, betas: tuple = (DEFAULT_BETA1, DEFAULT_BETA2), eps: float = DEFAULT_EPS, weight_decay: float = DEFAULT_WEIGHT_DECAY_ADAMW):
        """
        Initialize AdamW optimizer.
        """
        
        super().__init__(params)

        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.weight_decay = weight_decay

        # Initialize moment buffers (same as Adam)
        self.m_buffers = [None for _ in self.params]
        self.v_buffers = [None for _ in self.params]
        

###  AdamW Moment Updates - Same EMA, Different Context

In [16]:
#| export
class _AdamWUpdateMomentsMixin:
    """Mixin added to AdamW for moment updates."""
    def _update_moments(self, i: int, grad_data: np.ndarray) -> tuple:
        """
        Update first and second moment estimates with bias correction for AdamW.
        """
        
        # Initialize buffers if needed
        if self.m_buffers[i] is None:
            self.m_buffers[i] = np.zeros_like(grad_data)
            self.v_buffers[i] = np.zeros_like(grad_data)

        # Update biased first moment estimate
        self.m_buffers[i] = self.beta1 * self.m_buffers[i] + (1 - self.beta1) * grad_data

        # Update biased second moment estimate
        self.v_buffers[i] = self.beta2 * self.v_buffers[i] + (1 - self.beta2) * (grad_data ** 2)

        # Compute bias correction
        bias_correction1 = 1 - self.beta1 ** self.step_count
        bias_correction2 = 1 - self.beta2 ** self.step_count

        # Compute bias-corrected moments
        m_hat = self.m_buffers[i] / bias_correction1
        v_hat = self.v_buffers[i] / bias_correction2

        return m_hat, v_hat
        

# Attach _update_moments to AdamW
AdamW._update_moments = _AdamWUpdateMomentsMixin._update_moments

###  Unit Test: AdamW Moment Updates

In [17]:
def test_unit_adamw_update_moments():
    """ Test AdamW _update_moments produces correct bias-corrected moments."""
    print(" Unit Test: AdamW Moment Updates...")

    param = Tensor([1.0, 2.0], requires_grad=True)
    optimizer = AdamW([param], lr=0.01, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01)

    grad = np.array([0.1, 0.2])

    # Simulate step 1
    optimizer.step_count = 1
    m_hat, v_hat = optimizer._update_moments(0, grad)

    # At step 1, bias-corrected m_hat should equal the gradient
    assert np.allclose(m_hat, grad), f"m_hat should equal grad at step 1, got {m_hat}"
    assert np.allclose(v_hat, grad ** 2), f"v_hat should equal grad^2 at step 1, got {v_hat}"

    # Verify buffers were initialized
    assert optimizer.m_buffers[0] is not None
    assert optimizer.v_buffers[0] is not None

    # Verify AdamW and Adam produce same moment values for same input
    param_adam = Tensor([1.0, 2.0], requires_grad=True)
    adam_opt = Adam([param_adam], lr=0.01, betas=(0.9, 0.999), eps=1e-8)
    adam_opt.step_count = 1

    # Reset AdamW buffers for fair comparison
    param_adamw = Tensor([1.0, 2.0], requires_grad=True)
    adamw_opt = AdamW([param_adamw], lr=0.01, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01)
    adamw_opt.step_count = 1

    m_adam, v_adam = adam_opt._update_moments(0, grad)
    m_adamw, v_adamw = adamw_opt._update_moments(0, grad)

    assert np.allclose(m_adam, m_adamw), "Adam and AdamW moments should be identical"
    assert np.allclose(v_adam, v_adamw), "Adam and AdamW moments should be identical"

    print("✅ AdamW moment updates work correctly!")

if __name__ == "__main__":
    test_unit_adamw_update_moments()

 Unit Test: AdamW Moment Updates...
✅ AdamW moment updates work correctly!


###  AdamW Step - Decoupled Weight Decay Composition

In [19]:
#| export
class _AdamWStepMixin:
    """Mixin added to AdamW for step method."""
    def step(self):
        """
        Perform AdamW update step by composing helpers with decoupled weight decay.
        """
        
        # Increment step counter first
        self.step_count += 1

        for i, param in enumerate(self.params):
            if param.grad is None:
                continue

            # Extract gradient using shared helper
            grad_data = self._extract_gradient(param)

            # Update moments using PURE gradients (no weight decay mixed in)
            m_hat, v_hat = self._update_moments(i, grad_data)

            # Apply gradient-based update
            param.data = param.data - self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

            # Apply decoupled weight decay (separate from gradient update)
            if self.weight_decay != 0:
                param.data = param.data * (1 - self.lr * self.weight_decay)
        

# Attach step to AdamW
AdamW.step = _AdamWStepMixin.step

###  Unit Test: AdamW Optimizer

This test validates our AdamW implementation with decoupled weight decay.

In [21]:
def test_unit_adamw_optimizer():
    """ Test AdamW optimizer implementation."""
    print(" Unit Test: AdamW Optimizer...")

    # Test AdamW vs Adam difference in weight decay
    # Create identical parameters for comparison
    param_adam = Tensor([1.0, 2.0], requires_grad=True)
    param_adamw = Tensor([1.0, 2.0], requires_grad=True)

    # Create optimizers with same settings
    adam = Adam([param_adam], lr=0.01, weight_decay=0.01)
    adamw = AdamW([param_adamw], lr=0.01, weight_decay=0.01)

    # Set gradients AFTER creating optimizers (optimizer.__init__ resets grad to None)
    param_adam.grad = Tensor([0.1, 0.2])
    param_adamw.grad = Tensor([0.1, 0.2])

    # Take one step
    adam.step()
    adamw.step()

    # Results should be different due to weight decay implementation
    assert not np.allclose(param_adam.data, param_adamw.data, rtol=1e-6)
    
    # Test AdamW basic functionality
    param = Tensor([1.0, 2.0], requires_grad=True)
    optimizer = AdamW([param], lr=0.01, weight_decay=0.01)
    # Set gradient AFTER creating optimizer
    param.grad = Tensor([0.1, 0.2])
    original_data = param.data.copy()

    optimizer.step()

    # Parameter should have changed
    assert not np.array_equal(param.data, original_data)
    assert optimizer.step_count == 1

    # Test that moment buffers are created
    assert optimizer.m_buffers[0] is not None
    assert optimizer.v_buffers[0] is not None

    # Test zero weight decay behaves like Adam
    param1 = Tensor([1.0, 2.0], requires_grad=True)
    param2 = Tensor([1.0, 2.0], requires_grad=True)

    adam_no_wd = Adam([param1], lr=0.01, weight_decay=0.0)
    adamw_no_wd = AdamW([param2], lr=0.01, weight_decay=0.0)

    # Set gradients AFTER creating optimizers
    param1.grad = Tensor([0.1, 0.2])
    param2.grad = Tensor([0.1, 0.2])

    adam_no_wd.step()
    adamw_no_wd.step()

    # Should be very similar (within numerical precision)
    assert np.allclose(param1.data, param2.data, rtol=1e-10)

    print("✅ AdamW optimizer works correctly!")

if __name__ == "__main__":
    test_unit_adamw_optimizer()

 Unit Test: AdamW Optimizer...
✅ AdamW optimizer works correctly!


##  Systems Analysis: Optimizer Performance and Memory

In [23]:
def analyze_optimizer_memory_usage():
    """ Analyze memory usage of different optimizers."""
    print(" Analyzing Optimizer Memory Usage...")

    # Create test parameters of different sizes
    param_sizes = [1000, 10000, 100000]  # 1K, 10K, 100K parameters

    print("Optimizer Memory Analysis (per parameter tensor):")
    print("=" * 60)
    print(f"{'Size':<10} {'SGD':<10} {'Adam':<10} {'AdamW':<10} {'Ratio':<10}")
    print("-" * 60)

    for size in param_sizes:
        # Create parameter
        param = Tensor(rng.standard_normal(size), requires_grad=True)

        # SGD memory (parameter + momentum buffer)
        sgd = SGD([param], momentum=0.9)
        # Set gradient AFTER creating optimizer
        param.grad = Tensor(rng.standard_normal(size))
        sgd.step()  # Initialize buffers
        sgd_memory = size * 2  # param + momentum buffer

        # Adam memory (parameter + 2 moment buffers)
        param_adam = Tensor(rng.standard_normal(size), requires_grad=True)
        adam = Adam([param_adam])
        # Set gradient AFTER creating optimizer
        param_adam.grad = Tensor(rng.standard_normal(size))
        adam.step()  # Initialize buffers
        adam_memory = size * 3  # param + m_buffer + v_buffer

        # AdamW memory (same as Adam)
        adamw_memory = adam_memory

        # Memory ratio (Adam/SGD)
        ratio = adam_memory / sgd_memory

        print(f"{size:<10} {sgd_memory:<10} {adam_memory:<10} {adamw_memory:<10} {ratio:.1f}x")

    print("\n💡 Key Insights:")
    print("- SGD: 2× parameter memory (momentum buffer)")
    print("- Adam/AdamW: 3× parameter memory (two moment buffers)")
    print("- Memory scales linearly with model size")
    print("- Trade-off: More memory for better convergence")
analyze_optimizer_memory_usage()

 Analyzing Optimizer Memory Usage...
Optimizer Memory Analysis (per parameter tensor):
Size       SGD        Adam       AdamW      Ratio     
------------------------------------------------------------
1000       2000       3000       3000       1.5x
10000      20000      30000      30000      1.5x
100000     200000     300000     300000     1.5x

💡 Key Insights:
- SGD: 2× parameter memory (momentum buffer)
- Adam/AdamW: 3× parameter memory (two moment buffers)
- Memory scales linearly with model size
- Trade-off: More memory for better convergence


In [27]:
def analyze_optimizer_convergence_behavior():
    """ Analyze convergence behavior of different optimizers."""
    print(" Analyzing Optimizer Convergence Behavior...")

    # Simulate optimization of a quadratic function: f(x) = 0.5 * x^2
    # Optimal solution: x* = 0, gradient = x

    def quadratic_loss(x):
        """Simple quadratic function for optimization testing."""
        return 0.5 * (x ** 2).sum()

    def compute_gradient(x):
        """Gradient of quadratic function: df/dx = x."""
        return x.copy()

    # Starting point
    x_start = np.array([5.0, -3.0, 2.0])  # Far from optimum [0, 0, 0]

    # Test different optimizers
    optimizers_to_test = [
        ("SGD", SGD, {"lr": 0.1}),
        ("SGD+Momentum", SGD, {"lr": 0.1, "momentum": 0.9}),
        ("Adam", Adam, {"lr": 0.1}),
        ("AdamW", AdamW, {"lr": 0.1, "weight_decay": 0.01})
    ]

    print("Convergence Analysis (quadratic function f(x) = 0.5 * x²):")
    print("=" * 70)
    print(f"{'Optimizer':<15} {'Step 0':<12} {'Step 5':<12} {'Step 10':<12} {'Final Loss':<12}")
    print("-" * 70)

    for name, optimizer_class, kwargs in optimizers_to_test:
        # Reset parameter
        param = Tensor(x_start.copy(), requires_grad=True)
        optimizer = optimizer_class([param], **kwargs)

        losses = []

        # Run optimization for 10 steps
        for step in range(11):
            # Compute loss and gradient
            loss = quadratic_loss(param.data)
            param.grad = Tensor(compute_gradient(param.data))

            losses.append(loss)

            # Update parameters
            if step < 10:  # Don't update after last evaluation
                optimizer.step()
                optimizer.zero_grad()

        # Format results
        step0 = f"{losses[0]:.6f}"
        step5 = f"{losses[5]:.6f}"
        step10 = f"{losses[10]:.6f}"
        final = f"{losses[10]:.6f}"

        print(f"{name:<15} {step0:<12} {step5:<12} {step10:<12} {final:<12}")

    print("\n💡 Key Insights:")
    print("- SGD: Steady progress but can be slow")
    print("- SGD+Momentum: Faster convergence, less oscillation")
    print("- Adam: Adaptive rates help with different parameter scales")
    print("- AdamW: Similar to Adam with regularization effects")


##  Module Integration Test

Final validation that everything works together correctly.

In [31]:
def test_module():
    """ Module Test: Complete Integration

    Comprehensive test of entire module functionality.
    """
    print(" RUNNING MODULE INTEGRATION TEST")
    print("=" * 50)

    # Run all unit tests
    print("Running unit tests...")
    test_unit_extract_gradient()
    test_unit_optimizer_base()
    test_unit_sgd_optimizer()
    test_unit_adam_update_moments()
    test_unit_adam_optimizer()
    test_unit_adamw_update_moments()
    test_unit_adamw_optimizer()

    print("\nRunning integration scenarios...")

    # Test realistic neural network optimization scenario
    print(" Integration Test: Multi-layer Network Optimization...")

    # Import components from TinyTorch package (previous modules must be completed and exported)
    from tinytorch.foundation.layers import Linear
    from tinytorch.foundation.activations import ReLU
    from tinytorch.foundation.losses import MSELoss

    # Create parameters for a 2-layer network
    # Layer 1: 3 inputs -> 4 hidden
    W1 = Tensor(rng.standard_normal((3, 4)) * 0.1, requires_grad=True)
    b1 = Tensor(np.zeros(4), requires_grad=True)

    # Layer 2: 4 hidden -> 2 outputs
    W2 = Tensor(rng.standard_normal((4, 2)) * 0.1, requires_grad=True)
    b2 = Tensor(np.zeros(2), requires_grad=True)

    params = [W1, b1, W2, b2]

    # Test all optimizers on same network
    # Create optimizers BEFORE setting gradients (optimizer.__init__ resets grad to None)
    optimizers = [
        SGD(params, lr=0.01, momentum=0.9),
        Adam([p for p in params], lr=0.001),  # Fresh param list for Adam
        AdamW([p for p in params], lr=0.001, weight_decay=0.01)  # Fresh param list for AdamW
    ]

    # Add realistic gradients AFTER creating optimizers
    W1.grad = Tensor(rng.standard_normal((3, 4)) * 0.01)
    b1.grad = Tensor(rng.standard_normal(4) * 0.01)
    W2.grad = Tensor(rng.standard_normal((4, 2)) * 0.01)
    b2.grad = Tensor(rng.standard_normal(2) * 0.01)

    # Save original parameter values
    original_params = [p.data.copy() for p in params]

    # Test SGD
    optimizers[0].step()
    sgd_params = [p.data.copy() for p in params]

    # Restore parameters and test Adam
    for i, p in enumerate(params):
        p.data = original_params[i].copy()
        # Re-add gradients since they may have been modified
        if i == 0:
            p.grad = Tensor(rng.standard_normal((3, 4)) * 0.01)
        elif i == 1:
            p.grad = Tensor(rng.standard_normal(4) * 0.01)
        elif i == 2:
            p.grad = Tensor(rng.standard_normal((4, 2)) * 0.01)
        else:
            p.grad = Tensor(rng.standard_normal(2) * 0.01)

    # Update parameter references for Adam
    optimizers[1].params = params
    optimizers[1].step()
    adam_params = [p.data.copy() for p in params]

    # Restore parameters and test AdamW
    for i, p in enumerate(params):
        p.data = original_params[i].copy()
        # Re-add gradients
        if i == 0:
            p.grad = Tensor(rng.standard_normal((3, 4)) * 0.01)
        elif i == 1:
            p.grad = Tensor(rng.standard_normal(4) * 0.01)
        elif i == 2:
            p.grad = Tensor(rng.standard_normal((4, 2)) * 0.01)
        else:
            p.grad = Tensor(rng.standard_normal(2) * 0.01)

    # Update parameter references for AdamW
    optimizers[2].params = params
    optimizers[2].step()
    adamw_params = [p.data.copy() for p in params]

    # Verify parameters changed differently for each optimizer
    for i in range(len(params)):
        # Parameters should be different from original
        assert not np.array_equal(sgd_params[i], original_params[i])
        assert not np.array_equal(adam_params[i], original_params[i])
        assert not np.array_equal(adamw_params[i], original_params[i])

        # Different optimizers should produce different results
        assert not np.allclose(sgd_params[i], adam_params[i], rtol=1e-6)

    print(" Multi-layer network optimization works!")

    # Test optimizer state management
    print(" Integration Test: Optimizer State Management...")

    param = Tensor([1.0, 2.0], requires_grad=True)
    optimizer = Adam([param], lr=0.001)
    # Set gradient AFTER creating optimizer
    param.grad = Tensor([0.1, 0.2])

    # First step should initialize buffers
    optimizer.step()
    assert optimizer.m_buffers[0] is not None
    assert optimizer.v_buffers[0] is not None
    assert optimizer.step_count == 1

    # Zero grad should clear gradients but preserve optimizer state
    optimizer.zero_grad()
    assert param.grad is None
    assert optimizer.m_buffers[0] is not None  # State preserved
    assert optimizer.step_count == 1  # Step count preserved

    print("✅ Optimizer state management works!")

    print("\n" + "=" * 50)
    print(" ALL TESTS PASSED! Module ready for export.")


##  Aha Moment: Optimizers Update Weights

In [32]:
def demo_optimizers():
    """ See optimizers update weights."""
    print(" AHA MOMENT: Optimizers Update Weights")
    print("=" * 45)

    # Create a parameter with a gradient
    weight = Tensor(np.array([5.0]), requires_grad=True)

    # SGD takes a step in the opposite direction
    optimizer = SGD([weight], lr=0.5)
    # Set gradient AFTER creating optimizer (optimizer.__init__ resets grad to None)
    weight.grad = np.array([1.0])  # Gradient pointing "uphill"

    print(f"Initial weight: {weight.data[0]:.2f}")
    print(f"Gradient:       {weight.grad[0]:.2f} (pointing uphill)")

    optimizer.step()

    print(f"\nAfter SGD step: {weight.data[0]:.2f}")
    print(f"Moved: {5.0 - weight.data[0]:.2f} (opposite to gradient)")

    print("\n Optimizer moves weights to reduce loss!")

In [33]:
if __name__ == "__main__":
    test_module()
    print("\n")
    demo_optimizers()

 RUNNING MODULE INTEGRATION TEST
Running unit tests...
 Unit Test: Gradient Extraction...
✅ Gradient extraction works correctly!
 Unit Test: Base Optimizer...
✅ Base Optimizer works correctly!
 Unit Test: SGD Optimizer...
✅ SGD optimizer works correctly!
 Unit Test: Adam Moment Updates...
✅ Adam moment updates work correctly!
 Unit Test: Adam Optimizer...
✅ Adam optimizer works correctly!
 Unit Test: AdamW Moment Updates...
✅ AdamW moment updates work correctly!
 Unit Test: AdamW Optimizer...
✅ AdamW optimizer works correctly!

Running integration scenarios...
🧪 Integration Test: Multi-layer Network Optimization...
✅ Multi-layer network optimization works!
🧪 Integration Test: Optimizer State Management...
✅ Optimizer state management works!

 ALL TESTS PASSED! Module ready for export.


 AHA MOMENT: Optimizers Update Weights
Initial weight: 5.00
Gradient:       1.00 (pointing uphill)

After SGD step: 4.50
Moved: 0.50 (opposite to gradient)

 Optimizer moves weights to reduce loss!


In [35]:
from nbdev.export import nb_export
nb_export("optimizers.ipynb", lib_path='../tinytorch/foundation')